#  Detección de Tumores Cerebrales con YOLOv8

##  Descripción General
Este proyecto utiliza **YOLOv8** (You Only Look Once versión 8), un modelo de inteligencia artificial de detección de objetos, para identificar automáticamente tumores cerebrales en imágenes de resonancia magnética (MRI).

El modelo analiza una imagen MRI y determina si existe un tumor (clase **positive**) o no (clase **negative**), marcando con un recuadro (bounding box) la zona de interés.

---

##  Objetivo General
Desarrollar y entrenar un modelo de visión por computadora basado en YOLOv8 capaz de detectar tumores cerebrales en imágenes MRI, con el fin de servir como herramienta de apoyo al diagnóstico médico.

---

##  Planteamiento del Problema
El diagnóstico de tumores cerebrales es un proceso complejo y tardado que depende de la experiencia del médico radiólogo. Un error o retraso puede tener consecuencias graves para el paciente. La detección automatizada mediante inteligencia artificial puede:
- Reducir tiempos de diagnóstico
- Disminuir errores humanos
- Funcionar como segunda opinión para el médico
- Apoyar en zonas con escasez de especialistas

---

##  ¿Para qué se puede usar este modelo?
-  **Hospitales y clínicas:** como herramienta de apoyo al diagnóstico médico
-  **Investigación médica:** para estudiar patrones de tumores en grandes cantidades de imágenes
- **Zonas rurales o sin especialistas:** donde no hay radiólogos disponibles
- **Detección temprana:** identificar tumores en etapas tempranas cuando el tratamiento es más efectivo

---

##  ¿Cómo beneficia a la Ingeniería en Sistemas Computacionales?
Este proyecto conecta directamente con habilidades clave de la carrera:

| Área | Aplicación en este proyecto |
|------|-----------------------------|
| **Inteligencia Artificial** | Entrenamiento y evaluación de modelos YOLO |
| **Visión por Computadora** | Detección de objetos en imágenes médicas |
| **Cómputo en la Nube** | Uso de Google Colab con GPU y Roboflow |
| **Manejo de Datos** | Datasets, etiquetado, splits de entrenamiento |
| **Desarrollo de Software** | Pipeline completo: datos → modelo → inferencia |

---

##  ¿A quién beneficia?
- **Pacientes:** diagnóstico más rápido y preciso
- **Médicos y radiólogos:** herramienta de apoyo que reduce carga de trabajo
- **Hospitales:** optimización de recursos y tiempos
- **Ingenieros en Sistemas:** oportunidad laboral en el sector healthtech e IA médica
- **Investigadores:** base para desarrollar sistemas más avanzados de diagnóstico

---
## Celda 1 — Instalación de librerías
Antes de comenzar instalamos e importamos todas las herramientas que vamos a necesitar.

In [1]:
# Instalamos las librerías necesarias para el proyecto
# ultralytics: contiene el modelo YOLOv8
# roboflow: para descargar el dataset desde la nube
# split-folders: para dividir imágenes en carpetas
# pillow: para abrir y manipular imágenes
!pip install split-folders pillow ultralytics
!pip install roboflow

import numpy as np          # Para operaciones matemáticas con arreglos
import pandas as pd         # Para manejo de datos en tablas
import os                   # Para manejar rutas y carpetas del sistema
import matplotlib.pyplot as plt  # Para graficar y visualizar imágenes
from PIL import Image       # Para abrir y procesar imágenes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 97.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15


---
##  Celda 2 — Descarga del Dataset desde Roboflow
Descargamos automáticamente el dataset de tumores cerebrales desde Roboflow.

**Descripción del dataset:**
- **Nombre:** Brain Tumor Detection Dataset
- **Fuente:** Kaggle — Ultralytics
- **Total de imágenes:** 878 imágenes de resonancias magnéticas (MRI)
- **Clases:** `negative` (sin tumor) y `positive` (con tumor)
- **Formato:** YOLOv8 — imágenes .jpg con etiquetas .txt
- **División:** 70% entrenamiento | 20% validación | 10% prueba
- **Limpieza:** El dataset ya venía pre-limpiado y etiquetado por Ultralytics

In [2]:
!pip install roboflow

from roboflow import Roboflow

# Nos conectamos a Roboflow con nuestra API key personal
rf = Roboflow(api_key="DF4PBtFAKwo7rKfPhEru")

# Accedemos al workspace y al proyecto específico de tumores cerebrales
project = rf.workspace("trabajos-sandra").project("brain-tumor-3sjuy")

# Seleccionamos la versión 1 del dataset
version = project.version(1)

# Descargamos el dataset en formato YOLOv8 (imágenes + etiquetas + data.yaml)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to brain-tumor-1 in yolov8:: 100%|██████████| 1761/1761 [00:00<00:00, 11451.35it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


---
##  Celda 3 — Carga del modelo YOLOv8
Cargamos YOLOv8 en su versión **nano (n)**, que es la más ligera y rápida.
Este modelo ya fue pre-entrenado con millones de imágenes y lo vamos a re-entrenar (fine-tuning) con nuestro dataset de tumores.

In [3]:
from ultralytics import YOLO

# Cargamos el modelo YOLOv8 nano pre-entrenado
# yolov8n.pt = nano (más rápido) | yolov8s.pt = small | yolov8m.pt = medium
model = YOLO("yolov8n.pt")
#model = YOLO("yolov8s.pt")
#model = YOLO("yolov8m.pt")

---
## Celda 4 — Entrenamiento del modelo
Entrenamos el modelo con nuestro dataset. El modelo va a aprender a distinguir entre imágenes con tumor y sin tumor.

**Hiperparámetros:**
- `epochs=50`: el modelo revisa todas las imágenes 50 veces
- `batch=16`: procesa 16 imágenes a la vez
- `imgsz=640`: redimensiona todas las imágenes a 640x640 píxeles

Al terminar guarda el mejor modelo en `weights/best.pt`

In [4]:
# Entrenamos el modelo con nuestro dataset de tumores cerebrales
model.train(
    # Ruta al archivo de configuración del dataset (clases, rutas de imágenes)
    data='/content/brain-tumor-1',

    # Nombre del experimento y carpeta donde se guardan los resultados
    name="Deteccion Tumor Cerebral",
    project="Deteccion Tumor Cerebral",

    # Hiperparámetros del entrenamiento
    epochs=50,   # Número de veces que el modelo ve todo el dataset
    batch=16,    # Cantidad de imágenes procesadas al mismo tiempo
    imgsz=640,   # Tamaño al que se redimensionan las imágenes (640x640)
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/brain-tumor-1, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Deteccion Tumor Cerebral, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e5d044a20c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

---
## 🔍 Celda 5 — Inferencia (prueba con imagen nueva)
Cargamos el mejor modelo entrenado y lo probamos con una imagen MRI nueva que el modelo nunca ha visto.

El modelo regresa:
- **Clase:** 0 = negative (sin tumor) | 1 = positive (con tumor)
- **Confianza:** qué tan seguro está el modelo (ej. 69% = bastante seguro)
- **Bounding Box:** coordenadas del recuadro que marca la zona detectada

>  Al correr esta celda aparecerá un botón para seleccionar tu imagen MRI desde tu computadora. No necesitas hacer nada más.

In [6]:
from google.colab import files
from ultralytics import YOLO

# Subimos la imagen MRI desde nuestra computadora
# Al correr esta celda aparece un botón 'Elegir archivos' — selecciona cualquier imagen MRI
print(' Selecciona una imagen MRI para probar el modelo:')
img_upload = files.upload()
img_path = list(img_upload.keys())[0]  # Toma el nombre del archivo que subiste
print(f' Imagen cargada: {img_path}')

# Cargamos el mejor modelo guardado durante el entrenamiento
modelo_entrenado = YOLO("/content/runs/detect/Deteccion Tumor Cerebral/Deteccion Tumor Cerebral/weights/best.pt")

# Realizamos la predicción sobre la imagen subida
# save=True guarda la imagen con el bounding box dibujado
resultados = modelo_entrenado.predict(source=img_path, save=True)

# Recorremos los resultados e imprimimos la información de cada detección
for resultado in resultados:
    print(f"\nPredicciones para la imagen: {resultado.path}")

    # Por cada objeto detectado imprimimos su información
    for caja, conf, clase in zip(resultado.boxes.xyxy,   # Coordenadas del bounding box
                                  resultado.boxes.conf,   # Nivel de confianza (0 a 1)
                                  resultado.boxes.cls):   # Clase detectada (0=negative, 1=positive)
        nombre_clase = 'positive (con tumor)' if int(clase) == 1 else 'negative (sin tumor)'
        print(f"   Clase: {nombre_clase} | Confianza: {conf:.2%} | BBox: {caja.tolist()}")


📂 Selecciona una imagen MRI para probar el modelo:


Saving tt.jpg to tt.jpg
✅ Imagen cargada: tt.jpg

image 1/1 /content/tt.jpg: 448x640 1 1, 46.5ms
Speed: 2.4ms preprocess, 46.5ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict-2

Predicciones para la imagen: /content/tt.jpg
  🎯 Clase: positive (con tumor) | Confianza: 69.23% | BBox: [87.19290161132812, 23.90581703186035, 128.62777709960938, 73.95957946777344]


---
## Celda 6 — Descarga de resultados
Comprimimos y descargamos todos los resultados del entrenamiento:
- Gráficas de entrenamiento (curvas de pérdida, precisión, recall)
- Pesos del modelo entrenado (best.pt)
- Imágenes con las detecciones guardadas

In [ ]:
import shutil
from google.colab import files

# Ruta de la carpeta que contiene todos los resultados del entrenamiento
folder_path = "/content/runs"

# Nombre del archivo zip que se va a descargar
output_filename = "tumor_cerebral_M.zip"

# Comprimimos toda la carpeta en un archivo ZIP
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_path)

# Descargamos el archivo ZIP a nuestra computadora
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>